## Step 1: Load prepared data

Load the cleaned dataset (produced separately by scripts/clean_data.py)
as the starting point for model development.

In [1]:
import pandas as pd

df_clean = pd.read_csv("../data/item_clean.csv")
df_clean.shape

(954, 10)

## Step 2: Prepare features

Group rare locations (< 5 listings) into "Other" — with many unique
locations and under 1000 rows, most locations have too few examples
for a model to learn anything reliable from them individually.

In [2]:
location_counts = df_clean["location"].value_counts()
common_locations = location_counts[location_counts >= 5].index
df_clean["location"] = df_clean["location"].where(df_clean["location"].isin(common_locations), "Other")

df_clean["location"].nunique()

46

One-hot encode location: turn the single text column into N binary
columns (one per category), since models need numeric input and
location has no natural numeric order.

In [3]:
df_encoded = pd.get_dummies(df_clean, columns=["location"])
df_encoded.shape

(954, 55)

Split into features (X) and target (y). Exclude id (random, no signal),
price and price_per_m2 (these ARE the target or derived from it —
including them would leak the answer), currency and city (single
value across ~all rows, no signal).

In [4]:
features = df_encoded.drop(columns=["id", "price", "currency", "price_per_m2", "city"])
target = df_encoded["price"]

features.shape, target.shape

((954, 50), (954,))

Fill missing floor with median and missing hasRepair with the most
common value, since Linear Regression and some other models can't
handle NaN natively.

In [5]:
features["floor"] = features["floor"].fillna(features["floor"].median())
features["hasRepair"] = features["hasRepair"].fillna(features["hasRepair"].mode()[0])

features.isna().sum().sum()

np.int64(0)

Split into train/test sets (80/20). Random split avoids biased sets —
e.g. if data were sorted by price, a non-random split could put mostly
cheap listings in train and expensive ones in test. random_state=42
fixes the shuffle for reproducibility.

In [6]:
from sklearn.model_selection import train_test_split

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

features_train.shape, features_test.shape

((763, 50), (191, 50))

## Step 3: Choose evaluation metrics

Using three complementary metrics: MAE (average error in AZN, easy to
explain to non-technical stakeholders), RMSE (penalizes large errors
more heavily than MAE — reveals if a model has occasional very bad
predictions hidden behind a decent average), and R² (fraction of price
variance the model explains, 0 to 1 — a scale-independent measure of
overall fit).

In [7]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def evaluate(model, features_test, target_test):
    predictions = model.predict(features_test)
    return {
        "MAE": mean_absolute_error(target_test, predictions),
        "RMSE": root_mean_squared_error(target_test, predictions),
        "R2": r2_score(target_test, predictions),
    }

## Step 4: Compare candidate models

Train 4 candidate models covering different approaches: Linear
Regression (linear baseline), Ridge (linear + regularization, more
robust with many features), Random Forest (non-linear, tree ensemble),
Gradient Boosting (sequential tree ensemble, often strong out of the
box on tabular data).

In [8]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

candidates = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

Train each candidate and collect all three metrics into a comparison
table.

In [9]:
results = {}

for name, model in candidates.items():
    model.fit(features_train, target_train)
    results[name] = evaluate(model, features_test, target_test)

results_df = pd.DataFrame(results).T
results_df

,MAE,RMSE,R2
Linear Regression,85707.074924,167308.226901,0.668151
Ridge,86016.495429,174328.574290,0.639718
Random Forest,89527.460488,156051.224681,0.711304
Gradient Boosting,90922.145980,159481.270204,0.698474


### Metric interpretation

- **MAE** (average error in AZN) favors Linear Regression slightly —
  easiest to explain, but doesn't reveal occasional large misses.
- **RMSE** squares errors before averaging, so it penalizes large
  individual errors much more heavily than MAE. Random Forest has a
  noticeably smaller MAE-RMSE gap than Linear Regression (156k vs
  167k), suggesting Linear Regression has more outlier predictions
  hidden behind a decent average.
- **R²** measures the fraction of price variance explained by the
  model, relative to a naive "always predict the mean" baseline.
  Random Forest explains 71.1% of price variance vs 66.8% for Linear
  Regression — a meaningfully better overall fit.

**Decision: Random Forest.** It loses narrowly on MAE but wins clearly
on RMSE and R² — 2 out of 3 metrics — indicating more consistent
predictions with fewer large misses, which matters more for a
production model than a marginally better average error.

## Step 5: Improve the chosen model

Tune Random Forest hyperparameters via grid search with cross-validation,
searching over number of trees, tree depth, and min samples per split
to find a better configuration than the defaults used above.

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
)
grid_search.fit(features_train, target_train)

grid_search.best_params_

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

Evaluate the tuned model on the held-out test set with the same three
metrics, to see the actual improvement over the default configuration.

In [11]:
best_rf_model = grid_search.best_estimator_

tuned_results = evaluate(best_rf_model, features_test, target_test)
tuned_results

{'MAE': 88633.90294398738, 'RMSE': 153283.4165590071, 'R2': 0.7214545105067349}

Modest but consistent improvement across all three metrics after
tuning (200 trees vs default 100) — MAE -1%, RMSE -1.8%, R² +1.4pp.
Using this tuned model as final.

## Step 6: Save the final model

Save the chosen Random Forest model for use by the API — trained
above as part of the candidates comparison, no need to retrain.

In [12]:
import joblib

joblib.dump(best_rf_model, "../models/model.pkl")

['../models/model.pkl']